In [4]:
import sys
from pathlib import Path
import json
from collections import Counter
import geopandas as gpd
import numpy as np

# adiciona a raiz do projeto ao Python
sys.path.append(str(Path.cwd().parent))

from src.domain.enums import IncomeGroup, TravelMode, TripPurpose
from src.domain.agent import Agent
from src.simulation.population import generate_population
from src.simulation.origin_assignment import assign_origins
from src.simulation.purpose_choice import assign_purpose


In [100]:
import importlib
import src.simulation.purpose_choice as purpose_choice

importlib.reload(purpose_choice)

<module 'src.simulation.purpose_choice' from 'c:\\Users\\simon\\Documents\\GitHub\\EtPilot\\src\\simulation\\purpose_choice.py'>

In [64]:
# Testando enum
print(IncomeGroup.LOW)
print(IncomeGroup.LOW.value)

print(TravelMode.CAR)
print(TripPurpose.WORK)

print(IncomeGroup("low"))
print(TravelMode("bike"))
print(TripPurpose("education"))

IncomeGroup.LOW
low
TravelMode.CAR
TripPurpose.WORK
IncomeGroup.LOW
TravelMode.BIKE
TripPurpose.EDUCATION


In [65]:
# Testando agente
agent = Agent(
    agent_id=1,
    income_group=IncomeGroup.LOW
)

agent

Agent(agent_id=1, income_group=<IncomeGroup.LOW: 'low'>, origin_id=None, origin_node=None, purpose=None, destination_id=None, destination_node=None, mode=None, route_edges=[], travel_distance=None, travel_time=None)

In [66]:
print(agent.agent_id)
print(agent.income_group)
print(agent.income_group.value)
print(agent.origin_node)
print(agent.route_edges)

agent.origin_id = 15
agent.origin_node = 123456789

print(agent)

1
IncomeGroup.LOW
low
None
[]
Agent(agent_id=1, income_group=<IncomeGroup.LOW: 'low'>, origin_id=15, origin_node=123456789, purpose=None, destination_id=None, destination_node=None, mode=None, route_edges=[], travel_distance=None, travel_time=None)


Configura caminho para as pastas

In [67]:
PROJECT_ROOT = Path.cwd().parent

CONFIG_PATH = (
    PROJECT_ROOT
    / "config" / "config.json"
)

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)



    import json

# configuracoes dos agentes
CONFIG_AGENTS_PATH = (
    PROJECT_ROOT
    / "config"
    / "config_agents.json"
)

with open(
    CONFIG_AGENTS_PATH,
    "r",
    encoding="utf-8",
) as f:

    config_agents = json.load(f)

In [68]:
income_groups_config = (
    config["income"]["groups"]
)

income_shares = {
    IncomeGroup.LOW:
        income_groups_config["low"]["share"],

    IncomeGroup.MIDDLE:
        income_groups_config["middle"]["share"],

    IncomeGroup.HIGH:
        income_groups_config["high"]["share"],
}

income_shares

{<IncomeGroup.LOW: 'low'>: 0.20997644,
 <IncomeGroup.MIDDLE: 'middle'>: 0.61144735,
 <IncomeGroup.HIGH: 'high'>: 0.17857621}

Cria populacao de agentes

In [69]:
agents = generate_population(
    n_agents=1000,
    income_shares=income_shares,
    seed=42
)

In [70]:
counts = Counter(
    agent.income_group.value
    for agent in agents
)

counts

Counter({'middle': 611, 'low': 210, 'high': 179})

In [71]:
for group, count in counts.items():

    print(
        group,
        count,
        round(
            count / len(agents),
            4
        )
    )

high 179 0.179
low 210 0.21
middle 611 0.611


In [72]:
agents_42 = generate_population(
    n_agents=1000,
    income_shares=income_shares,
    seed=42
)

agents_123 = generate_population(
    n_agents=1000,
    income_shares=income_shares,
    seed=123
)

In [73]:
origins_path = (
    PROJECT_ROOT
    / config["paths"]["origins"]
)

origins = gpd.read_file(
    origins_path
)

print("CRS:", origins.crs)
print("Quantidade:", len(origins))
print("\nColunas:")
print(origins.columns.tolist())

display(
    origins.head()
)

CRS: EPSG:31982
Quantidade: 6189

Colunas:
['origin_id', 'ID_UNICO', 'nome_1KM', 'nome_5KM', 'nome_10KM', 'nome_50KM', 'nome_100KM', 'nome_500KM', 'QUADRANTE', 'MASC', 'FEM', 'POP', 'DOM_OCU', 'Shape_Leng', 'Shape_Area', 'node', 'geometry']


,origin_id,ID_UNICO,nome_1KM,nome_5KM,nome_10KM,nome_50KM,nome_100KM,nome_500KM,QUADRANTE,MASC,FEM,POP,DOM_OCU,Shape_Leng,Shape_Area,node,geometry
0,235.0,200ME52736N79976,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,26.0,29.0,55.0,16.0,0.007743,0.000004,2248442166,POINT (474386.204 6669552.438)
1,236.0,200ME52738N79976,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,50.0,55.0,105.0,30.0,0.007743,0.000004,296249745,POINT (474578.68 6669554.914)
2,237.0,200ME52736N79978,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,46.0,51.0,97.0,28.0,0.007743,0.000004,2248442166,POINT (474383.84 6669760.062)
3,238.0,200ME52738N79978,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,55.0,59.0,114.0,33.0,0.007743,0.000004,508060554,POINT (474576.318 6669762.539)
4,239.0,200ME52736N79980,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,37.0,43.0,80.0,23.0,0.007743,0.000004,583532859,POINT (474381.475 6669967.684)


In [74]:
print("CRS:")
print(origins.crs)

print("\nQuantidade de origens:")
print(len(origins))

print("\nTipo de geometria:")
print(origins.geometry.geom_type.value_counts())

print("\nColunas:")
print(origins.columns.tolist())

display(
    origins.head()
)

CRS:
EPSG:31982

Quantidade de origens:
6189

Tipo de geometria:
Point    6189
Name: count, dtype: int64

Colunas:
['origin_id', 'ID_UNICO', 'nome_1KM', 'nome_5KM', 'nome_10KM', 'nome_50KM', 'nome_100KM', 'nome_500KM', 'QUADRANTE', 'MASC', 'FEM', 'POP', 'DOM_OCU', 'Shape_Leng', 'Shape_Area', 'node', 'geometry']


,origin_id,ID_UNICO,nome_1KM,nome_5KM,nome_10KM,nome_50KM,nome_100KM,nome_500KM,QUADRANTE,MASC,FEM,POP,DOM_OCU,Shape_Leng,Shape_Area,node,geometry
0,235.0,200ME52736N79976,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,26.0,29.0,55.0,16.0,0.007743,0.000004,2248442166,POINT (474386.204 6669552.438)
1,236.0,200ME52738N79976,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,50.0,55.0,105.0,30.0,0.007743,0.000004,296249745,POINT (474578.68 6669554.914)
2,237.0,200ME52736N79978,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,46.0,51.0,97.0,28.0,0.007743,0.000004,2248442166,POINT (474383.84 6669760.062)
3,238.0,200ME52738N79978,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,55.0,59.0,114.0,33.0,0.007743,0.000004,508060554,POINT (474576.318 6669762.539)
4,239.0,200ME52736N79980,1KME5273N7997,5KME5270N7995,10KME5270N7990,50KME5250N7950,100KME5200N7950,500KME4800N7850,ID_14,37.0,43.0,80.0,23.0,0.007743,0.000004,583532859,POINT (474381.475 6669967.684)


In [75]:
origins.info()
origins.columns.tolist()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 6189 entries, 0 to 6188
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   origin_id   6189 non-null   float64 
 1   ID_UNICO    6189 non-null   str     
 2   nome_1KM    6189 non-null   str     
 3   nome_5KM    6189 non-null   str     
 4   nome_10KM   6189 non-null   str     
 5   nome_50KM   6189 non-null   str     
 6   nome_100KM  6189 non-null   str     
 7   nome_500KM  6189 non-null   str     
 8   QUADRANTE   6189 non-null   str     
 9   MASC        6189 non-null   float64 
 10  FEM         6189 non-null   float64 
 11  POP         6189 non-null   float64 
 12  DOM_OCU     6189 non-null   float64 
 13  Shape_Leng  6189 non-null   float64 
 14  Shape_Area  6189 non-null   float64 
 15  node        6189 non-null   int64   
 16  geometry    6189 non-null   geometry
dtypes: float64(7), geometry(1), int64(1), str(8)
memory usage: 1.4 MB


['origin_id',
 'ID_UNICO',
 'nome_1KM',
 'nome_5KM',
 'nome_10KM',
 'nome_50KM',
 'nome_100KM',
 'nome_500KM',
 'QUADRANTE',
 'MASC',
 'FEM',
 'POP',
 'DOM_OCU',
 'Shape_Leng',
 'Shape_Area',
 'node',
 'geometry']

In [76]:
origins_income_path = (
    PROJECT_ROOT
    / config["paths"]["origins_income"]
)

origins_income = gpd.read_file(
    origins_income_path
)

In [77]:
origins_income[
    [
        "origin_id",
        "POP",
        "node",
        "income_group",
    ]
].head()

,origin_id,POP,node,income_group
0,235.0,55.0,2248442166,NaN
1,236.0,105.0,296249745,high
2,237.0,97.0,2248442166,high
3,238.0,114.0,508060554,high
4,239.0,80.0,583532859,high


In [78]:
print(
    origins_income["income_group"]
    .value_counts(dropna=False)
)

income_group
middle    4022
low       1100
high       847
NaN        220
Name: count, dtype: int64


Atribuir as origens

In [79]:
agents = assign_origins(
    agents=agents,
    origins=origins_income,
    population_column="POP",
    seed=42,
)

In [80]:
for agent in agents[:10]:

    print(
        agent.agent_id,
        agent.income_group.value,
        agent.origin_id,
        agent.origin_node,
    )

1 high 5510.0 687045203
2 high 4097.0 443403302
3 high 6277.0 2504906641
4 high 4817.0 682212891
5 low 10422.0 3604184810
6 middle 9482.0 746978199
7 high 1448.0 8523144245
8 middle 6972.0 4263741567
9 low 11424.0 320803327
10 middle 2152.0 3557022274


Validações

In [81]:

print(
    "Agentes sem origin_id:",
    sum(
        agent.origin_id is None
        for agent in agents
    )
)

print(
    "Agentes sem origin_node:",
    sum(
        agent.origin_node is None
        for agent in agents
    )
)

validation = []

for agent in agents:

    origin_group = (
        origins_income
        .loc[
            origins_income["origin_id"]
            == agent.origin_id,
            "income_group"
        ]
        .iloc[0]
    )

    validation.append(
        agent.income_group.value
        == origin_group
    )

print(
    "Todos os agentes estão em origens do mesmo grupo:",
    all(validation)
)

print(
    "Incompatibilidades:",
    sum(
        not value
        for value in validation
    )
)

Agentes sem origin_id: 0
Agentes sem origin_node: 0
Todos os agentes estão em origens do mesmo grupo: True
Incompatibilidades: 0


Teste controlado

In [82]:
import pandas as pd

from collections import Counter
from src.domain.agent import Agent
from src.domain.enums import IncomeGroup
from src.simulation.origin_assignment import assign_origins


test_origins = pd.DataFrame(
    {
        "origin_id": [1, 2],
        "node": [101, 102],
        "income_group": [
            "low",
            "low",
        ],
        "POP": [
            100,
            900,
        ],
    }
)

test_agents = [
    Agent(
        agent_id=i,
        income_group=IncomeGroup.LOW,
    )
    for i in range(
        1,
        10001,
    )
]

test_agents = assign_origins(
    agents=test_agents,
    origins=test_origins,
    population_column="POP",
    seed=42,
)

In [83]:
test_counts = Counter(
    agent.origin_id
    for agent in test_agents
)

print(test_counts)

for origin_id, count in sorted(
    test_counts.items()
):
    print(
        origin_id,
        count,
        round(
            count / len(test_agents),
            3,
        ),
    )

Counter({np.int64(2): 9016, np.int64(1): 984})
1 984 0.098
2 9016 0.902


Definição das razões de viagem

In [84]:
test_purpose_probabilities = {
    TripPurpose.WORK: 0.20,
    TripPurpose.EDUCATION: 0.20,
    TripPurpose.HEALTH: 0.20,
    TripPurpose.SHOPPING: 0.20,
    TripPurpose.LEISURE: 0.20,
}

In [95]:
purpose_probabilities = (
    config_agents[
        "purpose_choice"
    ]
)

In [102]:
agents = purpose_choice.assign_purpose(
    agents=agents,
    purpose_probabilities=purpose_probabilities,
    seed=42,
)

print(purpose_probabilities)

{'low': {'work': 0.45, 'education': 0.15, 'shopping': 0.2, 'health': 0.1, 'leisure': 0.1}, 'middle': {'work': 0.35, 'education': 0.15, 'shopping': 0.2, 'health': 0.1, 'leisure': 0.2}, 'high': {'work': 0.3, 'education': 0.15, 'shopping': 0.2, 'health': 0.1, 'leisure': 0.25}}


In [103]:
for income_group in [
    "low",
    "middle",
    "high",
]:

    counts = Counter(
        agent.purpose.value
        for agent in agents
        if agent.income_group.value
        == income_group
    )

    print(
        income_group,
        counts
    )

low Counter({'work': 95, 'shopping': 42, 'education': 31, 'leisure': 21, 'health': 21})
middle Counter({'work': 214, 'leisure': 122, 'shopping': 122, 'education': 92, 'health': 61})
high Counter({'work': 53, 'leisure': 45, 'shopping': 36, 'education': 27, 'health': 18})
